# 기본 01 · Gray 변환과 Otsu 이진화

정상 문서 3장을 Gray와 Otsu 이진 영상으로 변환하고, 배열의 형태와 임계값을 확인합니다.

프로젝트 `.venv`의 NumPy, pandas, OpenCV, Matplotlib, IPython이 설치된 Python 커널에서 위에서 아래로 실행하세요. 결과는 메모리에서 계산합니다.

## 1. 입력과 처리 원리

입력은 `01_basic/doc01.png`~`doc03.png`입니다. `cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)`로 밝기 영상을 만들고, `cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)`로 임계값을 자동 선택합니다.

`threshold`의 첫 반환값은 선택한 임계값, 두 번째는 이진 영상입니다. Gray는 BGR 3채널을 1채널로 바꾸며, Otsu는 그 밝기 분포를 사용합니다.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

start = Path.cwd().resolve()
project = next((p for p in [start, *start.parents]
                if (p / "20261109/01_basic/01_preprocess.py").is_file()), None)
if project is None:
    raise FileNotFoundError("저장소 내부에서 실행하세요.")
print("OpenCV:", cv2.__version__)

exercise = project / "20261109/01_basic"

## 2. 실제 B01 실행과 수치 확인

`01_preprocess.py`의 B01 블록을 읽어 Gray 변환과 Otsu 이진화를 실행하고, 문서별 shape·임계값·픽셀 값을 표로 확인합니다.

In [ ]:
source = (exercise / "01_preprocess.py").read_text(encoding="utf-8")
block = source.split("# TODO B01", 1)[1].split("# TODO 끝 B01", 1)[0]
import textwrap
block = textwrap.dedent(block)
print(block.strip())
records, images = [], {}
for name in ["doc01", "doc02", "doc03"]:
    image = cv2.imread(str(exercise / f"{name}.png"))
    assert image is not None, name
    ns = {"cv2": cv2, "image": image}
    exec(compile(block, str(exercise / "01_preprocess.py"), "exec"), ns)
    gray, binary, threshold = ns["gray"], ns["binary"], ns["threshold"]
    images[name] = (image, gray, binary)
    records.append({"document": name, "input_shape": image.shape,
                    "gray_shape": gray.shape, "threshold": threshold,
                    "dtype": str(binary.dtype), "binary_values": np.unique(binary).tolist()})
display(pd.DataFrame(records))

## 3. 원본·Gray·이진 영상 비교

원본은 표시할 때 BGR를 RGB로 변환합니다. 같은 문서를 가로로 비교하며 Gray와 binary의 표시 범위는 0~255로 고정합니다.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for row, (name, (image, gray, binary)) in enumerate(images.items()):
    for col, (view, title) in enumerate([
        (cv2.cvtColor(image, cv2.COLOR_BGR2RGB), "Original"),
        (gray, "Gray"), (binary, "Otsu")]):
        axes[row, col].imshow(view, cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(f"{name}: {title}")
        axes[row, col].axis("off")
fig.tight_layout()
display(fig)
plt.close(fig)

## 4. 출력 계약과 독립 계산 검증

Gray의 shape·dtype, 이진 값의 범위를 검사하고 Otsu가 반환한 임계값을 직접 적용한 결과와 비교합니다. 이미 생성된 PNG가 있으면 재읽어 현재 계산 결과와 일치하는지도 확인합니다.

In [ ]:
for record in records:
    name = record["document"]
    image, gray, binary = images[name]
    assert gray.shape == image.shape[:2] == binary.shape
    assert gray.dtype == binary.dtype == np.uint8
    reference_gray = np.clip(np.rint(
        image[:, :, 0] * 0.114 + image[:, :, 1] * 0.587 + image[:, :, 2] * 0.299
    ), 0, 255).astype(np.uint8)
    assert np.abs(gray.astype(int) - reference_gray.astype(int)).max() <= 1
    assert set(np.unique(binary)).issubset({0, 255})
    reference_binary = np.where(gray > record["threshold"], 255, 0).astype(np.uint8)
    np.testing.assert_array_equal(binary, reference_binary)
    for kind, expected in [("gray", gray), ("binary", binary)]:
        path = exercise / f"p01_{name}_{kind}.png"
        if path.exists():
            np.testing.assert_array_equal(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), expected)
            print("기존 PNG 일치:", path.name)
        else:
            print("기존 PNG 없음, 메모리 계산만 검증:", path.name)
print("검증 통과: shape · dtype · Gray 기준 계산 · 임계값 적용")

## 5. 해석과 설명 질문

Gray 변환은 문서의 밝기 정보를 남기고, Otsu 이진화는 자동 임계값으로 검정·흰색을 나눕니다. README의 제작 환경 기준은 3문서 모두 `(700, 1000)`, 임계값 `145.0`입니다. 이 값은 참고값이며 위 표가 현재 환경의 계산 결과입니다.

**이진화 후 눈으로 읽기 쉬워지면 OCR 정확도도 반드시 높아질까요?** 반드시 높아지지는 않습니다. 이진화로 가는 획이 사라지거나 붙을 수 있으므로 같은 OCR 설정과 정답으로 별도 비교해야 합니다. 이번 작업은 영상 변환만 검증하며 OCR을 실행하거나 정확도를 측정하지 않습니다.